# BigData-Pass AI: Data-Driven Examiner

기출문제(JSON)와 암기 키트(PDF)를 결합하여 실제 시험과 유사한 변형 문제를 생성하는 AI CBT 시스템

**핵심 철학**: 정답 암기가 아니라, 출제 패턴과 사고 과정을 학습시킨다

## Cell 1: 환경 설정 및 라이브러리

In [ ]:
# 필수 패키지 설치 (최초 1회 실행)
# !pip install langchain langchain-openai langchain-community chromadb pypdf sentence-transformers pandas numpy matplotlib seaborn ipywidgets

In [ ]:
import os
import json
import random
from pathlib import Path
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, field

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document

import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")

In [ ]:
# OpenAI API 키 설정
# 방법 1: 직접 입력
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# 방법 2: 환경변수에서 로드
# API 키가 설정되어 있는지 확인
if "OPENAI_API_KEY" not in os.environ:
    print("[경고] OPENAI_API_KEY가 설정되지 않았습니다.")
    print("위 셀에서 API 키를 설정하거나, 환경변수로 설정해주세요.")
else:
    print("OpenAI API 키 설정 완료")

# 프로젝트 경로 설정
BASE_DIR = Path("/Users/honghyeongi/big")
DATA_DIR = BASE_DIR / "data"
JSON_DIR = DATA_DIR / "json"
PDF_PATH = DATA_DIR / "빅데이터암기키트.pdf"

print(f"Base Directory: {BASE_DIR}")
print(f"PDF Path: {PDF_PATH}")

## Cell 2: L1 Knowledge Core - 데이터 로드

In [ ]:
@dataclass
class ExamQuestion:
    """기출문제 데이터 클래스"""
    id: str
    subject: int
    subject_name: str
    question: str
    choices: List[str]
    answer: int
    keywords: List[str]
    round: int = 0
    
    def to_dict(self) -> Dict:
        return {
            "id": self.id,
            "subject": self.subject,
            "subject_name": self.subject_name,
            "question": self.question,
            "choices": self.choices,
            "answer": self.answer,
            "keywords": self.keywords
        }
    
    def get_correct_answer(self) -> str:
        return self.choices[self.answer - 1]
    
    def get_wrong_answers(self) -> List[str]:
        return [c for i, c in enumerate(self.choices) if i != self.answer - 1]

In [ ]:
class KnowledgeCore:
    """L1: Knowledge Core - 데이터 로드 및 관리"""
    
    def __init__(self, json_dir: Path, pdf_path: Path):
        self.json_dir = json_dir
        self.pdf_path = pdf_path
        self.exam_questions: List[ExamQuestion] = []
        self.pdf_documents: List[Document] = []
        self.exam_data_raw: List[Dict] = []
        
    def load_exam_json(self) -> List[ExamQuestion]:
        """기출문제 JSON 파일 로드"""
        json_files = sorted(self.json_dir.glob("exam_*.json"))
        print(f"발견된 기출문제 파일: {len(json_files)}개")
        
        for json_file in json_files:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                exam_round = data['exam_info']['round']
                self.exam_data_raw.append(data)
                
                for q in data['questions']:
                    exam_q = ExamQuestion(
                        id=q['id'],
                        subject=q['subject'],
                        subject_name=q['subject_name'],
                        question=q['question'],
                        choices=q['choices'],
                        answer=q['answer'],
                        keywords=q.get('keywords', []),
                        round=exam_round
                    )
                    self.exam_questions.append(exam_q)
        
        print(f"총 로드된 문제 수: {len(self.exam_questions)}개")
        return self.exam_questions
    
    def load_pdf(self, chunk_size: int = 500, chunk_overlap: int = 50) -> List[Document]:
        """암기키트 PDF 로드 및 청킹"""
        if not self.pdf_path.exists():
            print(f"[경고] PDF 파일을 찾을 수 없습니다: {self.pdf_path}")
            return []
        
        loader = PyPDFLoader(str(self.pdf_path))
        pages = loader.load()
        print(f"PDF 페이지 수: {len(pages)}")
        
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ".", " "]
        )
        
        self.pdf_documents = text_splitter.split_documents(pages)
        print(f"생성된 청크 수: {len(self.pdf_documents)}")
        return self.pdf_documents
    
    def get_questions_by_subject(self, subject: int) -> List[ExamQuestion]:
        """과목별 문제 필터링"""
        return [q for q in self.exam_questions if q.subject == subject]
    
    def get_questions_by_keyword(self, keyword: str) -> List[ExamQuestion]:
        """키워드로 문제 검색"""
        return [q for q in self.exam_questions 
                if keyword.lower() in [k.lower() for k in q.keywords] 
                or keyword.lower() in q.question.lower()]
    
    def get_statistics(self) -> Dict:
        """기출문제 통계"""
        stats = {
            "total_questions": len(self.exam_questions),
            "by_subject": Counter(q.subject for q in self.exam_questions),
            "by_round": Counter(q.round for q in self.exam_questions),
            "all_keywords": Counter(kw for q in self.exam_questions for kw in q.keywords)
        }
        return stats

In [ ]:
# Knowledge Core 초기화 및 데이터 로드
knowledge_core = KnowledgeCore(JSON_DIR, PDF_PATH)

# 기출문제 로드
exam_questions = knowledge_core.load_exam_json()

# PDF 로드
pdf_docs = knowledge_core.load_pdf()

# 통계 확인
stats = knowledge_core.get_statistics()
print(f"\n=== 기출문제 통계 ===")
print(f"총 문제 수: {stats['total_questions']}")
print(f"\n과목별 문제 수:")
subject_names = {1: "빅데이터 분석 기획", 2: "빅데이터 탐색", 3: "빅데이터 모델링", 4: "빅데이터 결과해석"}
for subj, count in sorted(stats['by_subject'].items()):
    print(f"  {subj}과목 ({subject_names.get(subj, '기타')}): {count}문제")

print(f"\n회차별 문제 수:")
for round_num, count in sorted(stats['by_round'].items()):
    print(f"  {round_num}회: {count}문제")

print(f"\n상위 10개 키워드:")
for kw, count in stats['all_keywords'].most_common(10):
    print(f"  {kw}: {count}회")

## Cell 3: Vector Store 구축 (RAG)

In [ ]:
class VectorStoreManager:
    """ChromaDB 기반 Vector Store 관리"""
    
    def __init__(self, persist_directory: str = "./chroma_db"):
        self.persist_directory = persist_directory
        self.embeddings = OpenAIEmbeddings()
        self.exam_vectorstore = None
        self.pdf_vectorstore = None
        
    def create_exam_vectorstore(self, questions: List[ExamQuestion]) -> Chroma:
        """기출문제 Vector Store 생성"""
        documents = []
        for q in questions:
            content = f"""
            [과목] {q.subject_name}
            [문제] {q.question}
            [보기]
            1. {q.choices[0]}
            2. {q.choices[1]}
            3. {q.choices[2]}
            4. {q.choices[3]}
            [정답] {q.answer}번
            [키워드] {', '.join(q.keywords)}
            """
            doc = Document(
                page_content=content,
                metadata={
                    "id": q.id,
                    "subject": q.subject,
                    "subject_name": q.subject_name,
                    "answer": q.answer,
                    "keywords": ",".join(q.keywords),
                    "round": q.round,
                    "type": "exam"
                }
            )
            documents.append(doc)
        
        self.exam_vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            collection_name="exam_questions",
            persist_directory=self.persist_directory
        )
        print(f"기출문제 Vector Store 생성 완료: {len(documents)}개 문서")
        return self.exam_vectorstore
    
    def create_pdf_vectorstore(self, documents: List[Document]) -> Chroma:
        """PDF 암기키트 Vector Store 생성"""
        for doc in documents:
            doc.metadata["type"] = "theory"
        
        self.pdf_vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            collection_name="pdf_theory",
            persist_directory=self.persist_directory
        )
        print(f"암기키트 Vector Store 생성 완료: {len(documents)}개 청크")
        return self.pdf_vectorstore
    
    def search_similar_questions(self, query: str, k: int = 5) -> List[Document]:
        """유사한 기출문제 검색"""
        if self.exam_vectorstore is None:
            raise ValueError("기출문제 Vector Store가 초기화되지 않았습니다.")
        return self.exam_vectorstore.similarity_search(query, k=k)
    
    def search_theory(self, query: str, k: int = 5) -> List[Document]:
        """관련 이론 검색"""
        if self.pdf_vectorstore is None:
            raise ValueError("암기키트 Vector Store가 초기화되지 않았습니다.")
        return self.pdf_vectorstore.similarity_search(query, k=k)
    
    def hybrid_search(self, query: str, k_exam: int = 3, k_theory: int = 3) -> Dict[str, List[Document]]:
        """기출 + 이론 하이브리드 검색"""
        return {
            "exam": self.search_similar_questions(query, k_exam),
            "theory": self.search_theory(query, k_theory)
        }

In [ ]:
# Vector Store 초기화 (API 키가 설정된 경우에만 실행)
vector_manager = None

if "OPENAI_API_KEY" in os.environ:
    vector_manager = VectorStoreManager(persist_directory=str(BASE_DIR / "chroma_db"))
    
    # 기출문제 Vector Store 생성
    vector_manager.create_exam_vectorstore(exam_questions)
    
    # PDF Vector Store 생성
    if pdf_docs:
        vector_manager.create_pdf_vectorstore(pdf_docs)
    
    print("\nVector Store 초기화 완료")
else:
    print("[경고] API 키가 설정되지 않아 Vector Store를 생성하지 않습니다.")

In [ ]:
# 검색 테스트
if vector_manager and vector_manager.exam_vectorstore:
    test_query = "혼동행렬 정밀도 재현율"
    results = vector_manager.hybrid_search(test_query)
    
    print(f"검색 쿼리: {test_query}")
    print("\n=== 유사 기출문제 ===")
    for doc in results["exam"][:2]:
        print(doc.page_content[:200])
        print("---")
    
    print("\n=== 관련 이론 ===")
    for doc in results["theory"][:2]:
        print(doc.page_content[:200])
        print("---")

## Cell 4: L2 Logic Engine - 출제 패턴 분석

In [ ]:
class PatternAnalyzer:
    """L2: Logic Engine - 출제 패턴 분석"""
    
    def __init__(self, questions: List[ExamQuestion]):
        self.questions = questions
        self.question_types = self._classify_question_types()
        self.keyword_patterns = self._analyze_keyword_patterns()
        
    def _classify_question_types(self) -> Dict[str, List[ExamQuestion]]:
        """문제 유형 분류"""
        types = {
            "definition": [],      # 정의/개념 문제
            "comparison": [],      # 비교 문제
            "calculation": [],     # 계산 문제
            "application": [],     # 응용/사례 문제
            "negative": [],        # 부정형 문제 (옳지 않은 것)
            "correct": [],         # 긍정형 문제 (옳은 것)
            "sequence": [],        # 순서/절차 문제
            "matching": []         # 연결/매칭 문제
        }
        
        negative_keywords = ["않은", "아닌", "틀린", "적절하지", "옳지"]
        calculation_keywords = ["계산", "구하", "값", "결과", "얼마"]
        comparison_keywords = ["차이", "비교", "다른", "같은"]
        
        for q in self.questions:
            question_text = q.question.lower()
            
            if any(kw in question_text for kw in negative_keywords):
                types["negative"].append(q)
            elif any(kw in question_text for kw in calculation_keywords):
                types["calculation"].append(q)
            elif any(kw in question_text for kw in comparison_keywords):
                types["comparison"].append(q)
            elif "옳은" in question_text or "알맞" in question_text:
                types["correct"].append(q)
            elif "순서" in question_text or "단계" in question_text:
                types["sequence"].append(q)
            else:
                types["definition"].append(q)
        
        return types
    
    def _analyze_keyword_patterns(self) -> Dict[str, Dict]:
        """키워드별 출제 패턴 분석"""
        patterns = defaultdict(lambda: {
            "count": 0,
            "subjects": [],
            "question_samples": [],
            "related_keywords": []
        })
        
        for q in self.questions:
            for kw in q.keywords:
                patterns[kw]["count"] += 1
                patterns[kw]["subjects"].append(q.subject)
                if len(patterns[kw]["question_samples"]) < 3:
                    patterns[kw]["question_samples"].append(q.question)
                patterns[kw]["related_keywords"].extend(
                    [k for k in q.keywords if k != kw]
                )
        
        return dict(patterns)
    
    def get_question_type_stats(self) -> pd.DataFrame:
        """문제 유형별 통계"""
        data = []
        for qtype, questions in self.question_types.items():
            data.append({
                "유형": qtype,
                "문제 수": len(questions),
                "비율": f"{len(questions) / len(self.questions) * 100:.1f}%"
            })
        return pd.DataFrame(data).sort_values("문제 수", ascending=False)
    
    def get_high_frequency_keywords(self, top_n: int = 20) -> List[tuple]:
        """고빈도 키워드 추출"""
        keyword_counts = [(kw, data["count"]) for kw, data in self.keyword_patterns.items()]
        return sorted(keyword_counts, key=lambda x: x[1], reverse=True)[:top_n]
    
    def suggest_question_style(self, keyword: str) -> Dict:
        """키워드 기반 출제 스타일 제안"""
        if keyword not in self.keyword_patterns:
            return {"error": "해당 키워드가 없습니다."}
        
        pattern = self.keyword_patterns[keyword]
        subject_counts = Counter(pattern["subjects"])
        main_subject = subject_counts.most_common(1)[0][0] if subject_counts else 1
        
        return {
            "keyword": keyword,
            "출제 빈도": pattern["count"],
            "주요 과목": main_subject,
            "예시 문제": pattern["question_samples"],
            "연관 키워드": list(set(pattern["related_keywords"]))[:5]
        }

In [ ]:
# 패턴 분석기 초기화
pattern_analyzer = PatternAnalyzer(exam_questions)

# 문제 유형 통계
print("=== 문제 유형별 통계 ===")
print(pattern_analyzer.get_question_type_stats())

# 고빈도 키워드
print("\n=== 고빈도 키워드 (Top 15) ===")
for kw, count in pattern_analyzer.get_high_frequency_keywords(15):
    print(f"  {kw}: {count}회")

# 특정 키워드 출제 패턴
print("\n=== '혼동행렬' 출제 패턴 ===")
style = pattern_analyzer.suggest_question_style("혼동행렬")
if "error" not in style:
    print(f"출제 빈도: {style['출제 빈도']}회")
    print(f"주요 과목: {style['주요 과목']}과목")
    print(f"연관 키워드: {style['연관 키워드']}")

In [ ]:
# 시각화: 과목별 문제 분포
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 과목별 분포
subject_counts = Counter(q.subject for q in exam_questions)
subjects = [f"{i}과목" for i in sorted(subject_counts.keys())]
counts = [subject_counts[i] for i in sorted(subject_counts.keys())]

axes[0].bar(subjects, counts, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])
axes[0].set_title("과목별 기출문제 분포")
axes[0].set_ylabel("문제 수")
for i, v in enumerate(counts):
    axes[0].text(i, v + 1, str(v), ha='center')

# 문제 유형 분포
type_stats = pattern_analyzer.get_question_type_stats()
axes[1].barh(type_stats["유형"], type_stats["문제 수"], color='#9b59b6')
axes[1].set_title("문제 유형별 분포")
axes[1].set_xlabel("문제 수")

plt.tight_layout()
plt.show()

## Cell 5: L3 Dynamic Generator - 통계 계산 엔진

In [ ]:
class DynamicGenerator:
    """L3: Dynamic Generator - 통계 계산 기반 문제 생성 (AI 환각 방지)"""
    
    @staticmethod
    def generate_confusion_matrix_problem() -> Dict:
        """혼동행렬 문제 생성"""
        TP = random.randint(30, 80)
        TN = random.randint(30, 80)
        FP = random.randint(5, 25)
        FN = random.randint(5, 25)
        
        # 정확한 계산
        precision = TP / (TP + FP)
        recall = TP / (TP + FN)
        accuracy = (TP + TN) / (TP + TN + FP + FN)
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        specificity = TN / (TN + FP)
        
        # 문제 유형 랜덤 선택
        problem_types = [
            {
                "question": f"""다음 분류 모델의 테스트 결과를 보고 정밀도(Precision)를 계산하시오.
                
                [혼동행렬]
                - True Positive (TP): {TP}
                - True Negative (TN): {TN}
                - False Positive (FP): {FP}
                - False Negative (FN): {FN}""",
                "correct": f"약 {precision:.2f}",
                "wrong": [
                    f"약 {recall:.2f}",
                    f"약 {accuracy:.2f}",
                    f"약 {specificity:.2f}"
                ],
                "answer_type": "precision",
                "calculation": f"Precision = TP/(TP+FP) = {TP}/({TP}+{FP}) = {precision:.4f}"
            },
            {
                "question": f"""다음 분류 모델의 테스트 결과를 보고 재현율(Recall)을 계산하시오.
                
                [혼동행렬]
                - True Positive (TP): {TP}
                - True Negative (TN): {TN}
                - False Positive (FP): {FP}
                - False Negative (FN): {FN}""",
                "correct": f"약 {recall:.2f}",
                "wrong": [
                    f"약 {precision:.2f}",
                    f"약 {accuracy:.2f}",
                    f"약 {f1_score:.2f}"
                ],
                "answer_type": "recall",
                "calculation": f"Recall = TP/(TP+FN) = {TP}/({TP}+{FN}) = {recall:.4f}"
            },
            {
                "question": f"""다음 분류 모델의 테스트 결과를 보고 F1-Score를 계산하시오.
                
                [혼동행렬]
                - True Positive (TP): {TP}
                - True Negative (TN): {TN}
                - False Positive (FP): {FP}
                - False Negative (FN): {FN}""",
                "correct": f"약 {f1_score:.2f}",
                "wrong": [
                    f"약 {precision:.2f}",
                    f"약 {recall:.2f}",
                    f"약 {accuracy:.2f}"
                ],
                "answer_type": "f1",
                "calculation": f"F1 = 2*(Precision*Recall)/(Precision+Recall) = 2*({precision:.4f}*{recall:.4f})/({precision:.4f}+{recall:.4f}) = {f1_score:.4f}"
            }
        ]
        
        problem = random.choice(problem_types)
        problem["stats"] = {
            "TP": TP, "TN": TN, "FP": FP, "FN": FN,
            "precision": precision, "recall": recall,
            "accuracy": accuracy, "f1_score": f1_score
        }
        problem["keywords"] = ["혼동행렬", "Confusion Matrix", problem["answer_type"]]
        
        return problem
    
    @staticmethod
    def generate_statistics_problem() -> Dict:
        """기술통계량 문제 생성"""
        # 데이터 생성
        data = np.random.randint(10, 100, size=random.randint(8, 15)).tolist()
        
        # 계산
        mean = np.mean(data)
        median = np.median(data)
        std = np.std(data, ddof=1)
        variance = np.var(data, ddof=1)
        data_range = max(data) - min(data)
        
        problem_types = [
            {
                "question": f"""다음 데이터의 평균을 계산하시오.
                
                데이터: {data}""",
                "correct": f"약 {mean:.2f}",
                "wrong": [
                    f"약 {median:.2f}",
                    f"약 {std:.2f}",
                    f"약 {float(data_range):.2f}"
                ],
                "answer_type": "mean",
                "calculation": f"평균 = 합계/개수 = {sum(data)}/{len(data)} = {mean:.4f}"
            },
            {
                "question": f"""다음 데이터의 표준편차(표본)를 계산하시오.
                
                데이터: {data}""",
                "correct": f"약 {std:.2f}",
                "wrong": [
                    f"약 {variance:.2f}",
                    f"약 {mean:.2f}",
                    f"약 {np.std(data):.2f}"  # 모표준편차 (오답)
                ],
                "answer_type": "std",
                "calculation": f"표본표준편차 = sqrt(표본분산) = sqrt({variance:.4f}) = {std:.4f}"
            }
        ]
        
        problem = random.choice(problem_types)
        problem["data"] = data
        problem["stats"] = {
            "mean": mean, "median": median, "std": std,
            "variance": variance, "range": data_range
        }
        problem["keywords"] = ["기술통계", "descriptive statistics", problem["answer_type"]]
        
        return problem
    
    @staticmethod
    def generate_probability_problem() -> Dict:
        """확률 문제 생성"""
        # 조건부 확률 문제
        P_A = round(random.uniform(0.3, 0.7), 2)
        P_B = round(random.uniform(0.3, 0.7), 2)
        P_A_and_B = round(min(P_A, P_B) * random.uniform(0.3, 0.8), 2)
        
        P_A_given_B = P_A_and_B / P_B if P_B > 0 else 0
        P_B_given_A = P_A_and_B / P_A if P_A > 0 else 0
        
        problem = {
            "question": f"""P(A) = {P_A}, P(B) = {P_B}, P(A and B) = {P_A_and_B}일 때,
            조건부 확률 P(A|B)를 계산하시오.""",
            "correct": f"약 {P_A_given_B:.2f}",
            "wrong": [
                f"약 {P_B_given_A:.2f}",
                f"약 {P_A_and_B:.2f}",
                f"약 {(P_A + P_B - P_A_and_B):.2f}"
            ],
            "answer_type": "conditional_probability",
            "calculation": f"P(A|B) = P(A and B)/P(B) = {P_A_and_B}/{P_B} = {P_A_given_B:.4f}",
            "stats": {
                "P_A": P_A, "P_B": P_B, "P_A_and_B": P_A_and_B,
                "P_A_given_B": P_A_given_B
            },
            "keywords": ["조건부확률", "베이즈정리", "conditional probability"]
        }
        
        return problem
    
    @staticmethod
    def generate_sampling_problem() -> Dict:
        """표본추출 문제 생성"""
        N = random.randint(500, 2000)  # 모집단 크기
        n = random.randint(50, 200)    # 표본 크기
        
        # 층화추출 관련
        strata = {
            "A": random.randint(100, 500),
            "B": random.randint(100, 500),
            "C": random.randint(100, 500)
        }
        total_N = sum(strata.values())
        proportional_samples = {k: round(v / total_N * n) for k, v in strata.items()}
        
        target_stratum = random.choice(list(strata.keys()))
        
        problem = {
            "question": f"""층화추출법을 사용하여 비례배분할 때, 층 {target_stratum}에서 추출해야 할 표본 수를 계산하시오.
            
            [층별 모집단]
            - 층 A: {strata['A']}명
            - 층 B: {strata['B']}명
            - 층 C: {strata['C']}명
            
            전체 표본 크기: {n}명""",
            "correct": f"{proportional_samples[target_stratum]}명",
            "wrong": [
                f"{proportional_samples[k]}명" for k in strata.keys() if k != target_stratum
            ] + [f"{n // 3}명"],
            "answer_type": "stratified_sampling",
            "calculation": f"층 {target_stratum} 표본 수 = ({strata[target_stratum]}/{total_N}) * {n} = {proportional_samples[target_stratum]}",
            "stats": {
                "strata": strata,
                "total_N": total_N,
                "n": n,
                "proportional_samples": proportional_samples
            },
            "keywords": ["층화추출", "비례배분", "stratified sampling"]
        }
        
        return problem
    
    def generate_random_problem(self) -> Dict:
        """랜덤 문제 생성"""
        generators = [
            self.generate_confusion_matrix_problem,
            self.generate_statistics_problem,
            self.generate_probability_problem,
            self.generate_sampling_problem
        ]
        return random.choice(generators)()

In [ ]:
# Dynamic Generator 테스트
generator = DynamicGenerator()

print("=== 혼동행렬 문제 예시 ===")
cm_problem = generator.generate_confusion_matrix_problem()
print(cm_problem["question"])
print(f"\n정답: {cm_problem['correct']}")
print(f"계산 과정: {cm_problem['calculation']}")

print("\n=== 통계 문제 예시 ===")
stat_problem = generator.generate_statistics_problem()
print(stat_problem["question"])
print(f"\n정답: {stat_problem['correct']}")
print(f"계산 과정: {stat_problem['calculation']}")

## Cell 6: RAG 기반 문제 생성 모델

In [ ]:
class QuestionGenerator:
    """RAG + LLM 기반 문제 생성기"""
    
    def __init__(self, vector_manager: VectorStoreManager, pattern_analyzer: PatternAnalyzer):
        self.vector_manager = vector_manager
        self.pattern_analyzer = pattern_analyzer
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
        self.dynamic_gen = DynamicGenerator()
        
    def _get_context(self, topic: str) -> Dict[str, str]:
        """RAG를 통한 컨텍스트 검색"""
        results = self.vector_manager.hybrid_search(topic, k_exam=3, k_theory=3)
        
        exam_context = "\n\n".join([doc.page_content for doc in results["exam"]])
        theory_context = "\n\n".join([doc.page_content for doc in results["theory"]])
        
        return {
            "exam": exam_context,
            "theory": theory_context
        }
    
    def generate_concept_question(self, topic: str, subject: int = 1) -> Dict:
        """개념 기반 문제 생성"""
        context = self._get_context(topic)
        pattern = self.pattern_analyzer.suggest_question_style(topic)
        
        prompt = PromptTemplate(
            input_variables=["topic", "exam_context", "theory_context", "subject"],
            template="""
당신은 빅데이터분석기사 시험 출제위원입니다.
아래 제공된 [기출문제]와 [이론]을 참고하여 새로운 4지선다형 문제를 출제하세요.

[주제]: {topic}
[과목]: {subject}과목

[기출문제 예시]
{exam_context}

[관련 이론]
{theory_context}

다음 JSON 형식으로 출력하세요:
{{
    "question": "문제 텍스트",
    "choices": ["보기1", "보기2", "보기3", "보기4"],
    "answer": 정답번호(1-4),
    "keywords": ["키워드1", "키워드2"],
    "explanation": {{
        "problem_analysis": "문제 분석",
        "theory_basis": "이론 근거",
        "answer_derivation": "정답 도출 과정",
        "wrong_analysis": "오답 분석"
    }}
}}

중요:
- 기출문제와 동일하지 않은 새로운 문제를 생성하세요
- 오답도 그럴듯하게 만들어 학습 효과를 높이세요
- 난이도는 실제 시험과 유사하게 조절하세요
"""
        )
        
        response = self.llm.invoke(prompt.format(
            topic=topic,
            exam_context=context["exam"],
            theory_context=context["theory"],
            subject=subject
        ))
        
        # JSON 파싱
        try:
            content = response.content
            # JSON 블록 추출
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0]
            elif "```" in content:
                content = content.split("```")[1].split("```")[0]
            
            result = json.loads(content.strip())
            result["subject"] = subject
            result["topic"] = topic
            result["type"] = "concept"
            return result
        except json.JSONDecodeError:
            return {
                "error": "JSON 파싱 실패",
                "raw_response": response.content
            }
    
    def generate_calculation_question(self, problem_type: str = "confusion_matrix") -> Dict:
        """계산 문제 생성 (Dynamic Generator 활용)"""
        if problem_type == "confusion_matrix":
            problem = self.dynamic_gen.generate_confusion_matrix_problem()
        elif problem_type == "statistics":
            problem = self.dynamic_gen.generate_statistics_problem()
        elif problem_type == "probability":
            problem = self.dynamic_gen.generate_probability_problem()
        elif problem_type == "sampling":
            problem = self.dynamic_gen.generate_sampling_problem()
        else:
            problem = self.dynamic_gen.generate_random_problem()
        
        # 보기 섞기
        choices = [problem["correct"]] + problem["wrong"][:3]
        random.shuffle(choices)
        answer = choices.index(problem["correct"]) + 1
        
        return {
            "question": problem["question"],
            "choices": choices,
            "answer": answer,
            "keywords": problem.get("keywords", []),
            "explanation": {
                "problem_analysis": f"{problem['answer_type']} 계산 문제",
                "theory_basis": problem.get("calculation", ""),
                "answer_derivation": f"정답: {problem['correct']}",
                "wrong_analysis": f"오답 보기: {problem['wrong']}"
            },
            "type": "calculation",
            "stats": problem.get("stats", {})
        }
    
    def generate_mixed_exam(self, n_questions: int = 10, subject: Optional[int] = None) -> List[Dict]:
        """혼합 모의고사 생성"""
        questions = []
        
        # 문제 유형 비율: 개념 60%, 계산 40%
        n_concept = int(n_questions * 0.6)
        n_calculation = n_questions - n_concept
        
        # 고빈도 키워드에서 주제 선택
        top_keywords = [kw for kw, _ in self.pattern_analyzer.get_high_frequency_keywords(20)]
        
        # 개념 문제 생성
        for _ in range(n_concept):
            topic = random.choice(top_keywords)
            subj = subject if subject else random.randint(1, 4)
            try:
                q = self.generate_concept_question(topic, subj)
                if "error" not in q:
                    questions.append(q)
            except Exception as e:
                print(f"개념 문제 생성 실패: {e}")
        
        # 계산 문제 생성
        calc_types = ["confusion_matrix", "statistics", "probability", "sampling"]
        for _ in range(n_calculation):
            calc_type = random.choice(calc_types)
            try:
                q = self.generate_calculation_question(calc_type)
                questions.append(q)
            except Exception as e:
                print(f"계산 문제 생성 실패: {e}")
        
        # 문제 번호 부여
        for i, q in enumerate(questions):
            q["id"] = f"GEN-{i+1:02d}"
        
        random.shuffle(questions)
        return questions

In [ ]:
# Question Generator 초기화 및 테스트
if vector_manager:
    q_generator = QuestionGenerator(vector_manager, pattern_analyzer)
    
    # 개념 문제 생성 테스트
    print("=== 개념 문제 생성 테스트 ===")
    concept_q = q_generator.generate_concept_question("빅데이터 3V", subject=1)
    if "error" not in concept_q:
        print(f"문제: {concept_q['question']}")
        print(f"보기:")
        for i, c in enumerate(concept_q['choices'], 1):
            print(f"  {i}. {c}")
        print(f"정답: {concept_q['answer']}번")
    else:
        print(concept_q)
    
    print("\n=== 계산 문제 생성 테스트 ===")
    calc_q = q_generator.generate_calculation_question("confusion_matrix")
    print(f"문제: {calc_q['question']}")
    print(f"보기:")
    for i, c in enumerate(calc_q['choices'], 1):
        print(f"  {i}. {c}")
    print(f"정답: {calc_q['answer']}번")
else:
    print("[경고] Vector Manager가 초기화되지 않았습니다. API 키를 설정해주세요.")
    
    # API 없이도 계산 문제는 생성 가능
    print("\n=== 계산 문제 (API 불필요) ===")
    calc_q = DynamicGenerator().generate_random_problem()
    print(f"문제: {calc_q['question']}")
    print(f"정답: {calc_q['correct']}")

## Cell 7: 오답 설계 엔진

In [ ]:
class DistractorEngine:
    """오답 설계 엔진 - 매력적인 오답 생성"""
    
    def __init__(self, questions: List[ExamQuestion]):
        self.questions = questions
        self.distractor_patterns = self._analyze_distractor_patterns()
        
    def _analyze_distractor_patterns(self) -> Dict:
        """기출문제에서 오답 패턴 분석"""
        patterns = {
            "similar_terms": [],      # 유사 용어
            "partial_correct": [],    # 부분적으로 맞는 답
            "calculation_error": [],  # 계산 실수 유도
            "concept_confusion": []   # 개념 혼동
        }
        
        for q in self.questions:
            correct = q.get_correct_answer()
            wrongs = q.get_wrong_answers()
            
            for wrong in wrongs:
                # 유사 용어 패턴
                if len(set(correct.split()) & set(wrong.split())) > 0:
                    patterns["similar_terms"].append({
                        "correct": correct,
                        "wrong": wrong,
                        "keywords": q.keywords
                    })
        
        return patterns
    
    def generate_distractors(self, correct_answer: str, topic: str, n: int = 3) -> List[str]:
        """매력적인 오답 생성"""
        distractors = []
        
        # 패턴 기반 오답 생성
        similar = [p for p in self.distractor_patterns["similar_terms"] 
                   if any(kw in topic for kw in p.get("keywords", []))]
        
        if similar:
            for p in random.sample(similar, min(n, len(similar))):
                distractors.append(p["wrong"])
        
        return distractors[:n]
    
    def evaluate_distractor_quality(self, question: Dict) -> Dict:
        """오답 품질 평가"""
        correct = question["choices"][question["answer"] - 1]
        wrongs = [c for i, c in enumerate(question["choices"]) if i != question["answer"] - 1]
        
        scores = []
        for wrong in wrongs:
            # 길이 유사도
            len_score = 1 - abs(len(correct) - len(wrong)) / max(len(correct), len(wrong))
            
            # 단어 유사도
            correct_words = set(correct.split())
            wrong_words = set(wrong.split())
            word_score = len(correct_words & wrong_words) / max(len(correct_words), 1)
            
            scores.append({
                "distractor": wrong,
                "length_similarity": len_score,
                "word_overlap": word_score,
                "overall": (len_score + word_score) / 2
            })
        
        return {
            "correct_answer": correct,
            "distractor_scores": scores,
            "average_quality": np.mean([s["overall"] for s in scores])
        }

In [ ]:
# 오답 설계 엔진 테스트
distractor_engine = DistractorEngine(exam_questions)

print("=== 기출문제 오답 품질 분석 ===")
sample_questions = random.sample(exam_questions, 3)

for q in sample_questions:
    analysis = distractor_engine.evaluate_distractor_quality(q.to_dict())
    print(f"\n문제: {q.question[:50]}...")
    print(f"정답: {analysis['correct_answer']}")
    print(f"오답 품질 점수: {analysis['average_quality']:.2f}")

## Cell 8: CBT 시스템 및 약점 분석

In [ ]:
@dataclass
class TestSession:
    """시험 세션 관리"""
    session_id: str
    questions: List[Dict] = field(default_factory=list)
    answers: Dict[str, int] = field(default_factory=dict)
    results: Dict[str, bool] = field(default_factory=dict)
    start_time: float = 0
    end_time: float = 0
    
    def submit_answer(self, question_id: str, answer: int):
        self.answers[question_id] = answer
    
    def grade(self):
        for q in self.questions:
            qid = q["id"]
            if qid in self.answers:
                self.results[qid] = self.answers[qid] == q["answer"]
    
    def get_score(self) -> Dict:
        correct = sum(self.results.values())
        total = len(self.questions)
        return {
            "correct": correct,
            "total": total,
            "percentage": (correct / total * 100) if total > 0 else 0
        }

In [ ]:
class CBTSystem:
    """CBT (Computer Based Test) 시스템"""
    
    def __init__(self):
        self.sessions: Dict[str, TestSession] = {}
        self.history: List[TestSession] = []
        
    def create_session(self, questions: List[Dict]) -> TestSession:
        """새 시험 세션 생성"""
        import time
        session_id = f"SESSION_{int(time.time())}"
        session = TestSession(
            session_id=session_id,
            questions=questions,
            start_time=time.time()
        )
        self.sessions[session_id] = session
        return session
    
    def display_question(self, session: TestSession, q_index: int):
        """문제 출력"""
        if q_index >= len(session.questions):
            print("모든 문제를 완료했습니다.")
            return
        
        q = session.questions[q_index]
        print(f"\n{'='*50}")
        print(f"문제 {q_index + 1}/{len(session.questions)}")
        print(f"{'='*50}")
        print(f"\n{q['question']}\n")
        
        for i, choice in enumerate(q['choices'], 1):
            print(f"  {i}. {choice}")
        print()
    
    def submit_and_show_result(self, session: TestSession, q_index: int, answer: int):
        """답안 제출 및 결과 표시"""
        q = session.questions[q_index]
        session.submit_answer(q["id"], answer)
        
        is_correct = answer == q["answer"]
        session.results[q["id"]] = is_correct
        
        if is_correct:
            print("[정답입니다!]")
        else:
            print(f"[오답입니다. 정답: {q['answer']}번]")
        
        # 해설 표시
        if "explanation" in q:
            exp = q["explanation"]
            print(f"\n--- 해설 ---")
            print(f"1. 문제 분석: {exp.get('problem_analysis', 'N/A')}")
            print(f"2. 이론 근거: {exp.get('theory_basis', 'N/A')}")
            print(f"3. 정답 도출: {exp.get('answer_derivation', 'N/A')}")
            print(f"4. 오답 분석: {exp.get('wrong_analysis', 'N/A')}")
    
    def finish_session(self, session: TestSession) -> Dict:
        """세션 종료 및 결과 반환"""
        import time
        session.end_time = time.time()
        session.grade()
        self.history.append(session)
        
        score = session.get_score()
        duration = session.end_time - session.start_time
        
        return {
            "session_id": session.session_id,
            "score": score,
            "duration_seconds": duration,
            "wrong_questions": [q for q in session.questions 
                               if not session.results.get(q["id"], False)]
        }

In [ ]:
class WeaknessAnalyzer:
    """약점 분석 시스템"""
    
    def __init__(self):
        self.answer_history: List[Dict] = []
        
    def record_answer(self, question: Dict, is_correct: bool, time_spent: float = 0):
        """답안 기록"""
        self.answer_history.append({
            "question_id": question.get("id", "unknown"),
            "keywords": question.get("keywords", []),
            "subject": question.get("subject", 0),
            "type": question.get("type", "concept"),
            "is_correct": is_correct,
            "time_spent": time_spent
        })
    
    def analyze_by_keyword(self) -> pd.DataFrame:
        """키워드별 정답률 분석"""
        keyword_stats = defaultdict(lambda: {"correct": 0, "total": 0})
        
        for record in self.answer_history:
            for kw in record["keywords"]:
                keyword_stats[kw]["total"] += 1
                if record["is_correct"]:
                    keyword_stats[kw]["correct"] += 1
        
        data = []
        for kw, stats in keyword_stats.items():
            if stats["total"] >= 2:  # 최소 2문제 이상
                data.append({
                    "키워드": kw,
                    "총 문제수": stats["total"],
                    "정답": stats["correct"],
                    "정답률": stats["correct"] / stats["total"] * 100
                })
        
        return pd.DataFrame(data).sort_values("정답률")
    
    def analyze_by_subject(self) -> pd.DataFrame:
        """과목별 정답률 분석"""
        subject_stats = defaultdict(lambda: {"correct": 0, "total": 0})
        
        for record in self.answer_history:
            subj = record["subject"]
            subject_stats[subj]["total"] += 1
            if record["is_correct"]:
                subject_stats[subj]["correct"] += 1
        
        subject_names = {
            1: "빅데이터 분석 기획",
            2: "빅데이터 탐색",
            3: "빅데이터 모델링",
            4: "빅데이터 결과해석"
        }
        
        data = []
        for subj, stats in subject_stats.items():
            data.append({
                "과목": subject_names.get(subj, f"{subj}과목"),
                "총 문제수": stats["total"],
                "정답": stats["correct"],
                "정답률": stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
            })
        
        return pd.DataFrame(data).sort_values("정답률")
    
    def get_weak_areas(self, threshold: float = 60.0) -> List[str]:
        """취약 영역 추출 (정답률 threshold% 미만)"""
        keyword_df = self.analyze_by_keyword()
        weak = keyword_df[keyword_df["정답률"] < threshold]
        return weak["키워드"].tolist()
    
    def generate_study_recommendation(self) -> str:
        """학습 추천 생성"""
        weak_areas = self.get_weak_areas()
        subject_df = self.analyze_by_subject()
        
        recommendation = "[학습 추천]\n\n"
        
        if len(weak_areas) > 0:
            recommendation += "1. 취약 키워드 (집중 학습 필요):\n"
            for kw in weak_areas[:5]:
                recommendation += f"   - {kw}\n"
        
        if len(subject_df) > 0:
            weakest_subject = subject_df.iloc[0]
            recommendation += f"\n2. 가장 취약한 과목: {weakest_subject['과목']} (정답률: {weakest_subject['정답률']:.1f}%)\n"
        
        return recommendation
    
    def plot_analysis(self):
        """분석 결과 시각화"""
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # 과목별 정답률
        subject_df = self.analyze_by_subject()
        if len(subject_df) > 0:
            colors = ['#e74c3c' if x < 60 else '#2ecc71' for x in subject_df['정답률']]
            axes[0].barh(subject_df['과목'], subject_df['정답률'], color=colors)
            axes[0].axvline(x=60, color='gray', linestyle='--', label='합격선 (60%)')
            axes[0].set_xlabel('정답률 (%)')
            axes[0].set_title('과목별 정답률')
            axes[0].legend()
        
        # 키워드별 정답률 (하위 10개)
        keyword_df = self.analyze_by_keyword()
        if len(keyword_df) > 0:
            bottom_10 = keyword_df.head(10)
            colors = ['#e74c3c' if x < 60 else '#f39c12' for x in bottom_10['정답률']]
            axes[1].barh(bottom_10['키워드'], bottom_10['정답률'], color=colors)
            axes[1].axvline(x=60, color='gray', linestyle='--')
            axes[1].set_xlabel('정답률 (%)')
            axes[1].set_title('취약 키워드 Top 10')
        
        plt.tight_layout()
        plt.show()

In [ ]:
# CBT 시스템 데모
cbt = CBTSystem()
weakness_analyzer = WeaknessAnalyzer()

# 샘플 문제 생성 (Dynamic Generator 활용)
demo_questions = []
generator = DynamicGenerator()

for i in range(5):
    problem = generator.generate_random_problem()
    choices = [problem["correct"]] + problem["wrong"][:3]
    random.shuffle(choices)
    answer = choices.index(problem["correct"]) + 1
    
    demo_questions.append({
        "id": f"DEMO-{i+1:02d}",
        "question": problem["question"],
        "choices": choices,
        "answer": answer,
        "keywords": problem.get("keywords", []),
        "subject": 4,  # 결과해석
        "type": "calculation",
        "explanation": {
            "problem_analysis": f"{problem['answer_type']} 문제",
            "theory_basis": problem.get("calculation", ""),
            "answer_derivation": f"정답: {problem['correct']}",
            "wrong_analysis": "오답 보기는 유사한 값이지만 계산 방식이 다름"
        }
    })

# 세션 생성
session = cbt.create_session(demo_questions)
print(f"세션 생성: {session.session_id}")
print(f"총 문제 수: {len(session.questions)}개\n")

In [ ]:
# 대화형 CBT 테스트 (실제 사용시)
def run_cbt_demo(session: TestSession, cbt: CBTSystem, analyzer: WeaknessAnalyzer):
    """CBT 데모 실행"""
    print("[CBT 시험 시작]")
    print("각 문제에 대해 1-4 중 하나를 입력하세요.\n")
    
    for i, q in enumerate(session.questions):
        cbt.display_question(session, i)
        
        # 데모용: 랜덤 답안 제출
        user_answer = random.randint(1, 4)
        print(f"[자동 답안 제출: {user_answer}]")
        
        cbt.submit_and_show_result(session, i, user_answer)
        
        # 약점 분석기에 기록
        analyzer.record_answer(
            q, 
            is_correct=(user_answer == q["answer"])
        )
        print()
    
    # 최종 결과
    result = cbt.finish_session(session)
    print("\n" + "="*50)
    print("[시험 종료]")
    print(f"점수: {result['score']['correct']}/{result['score']['total']} ({result['score']['percentage']:.1f}%)")
    print(f"소요 시간: {result['duration_seconds']:.1f}초")
    
    return result

# 데모 실행
result = run_cbt_demo(session, cbt, weakness_analyzer)

In [ ]:
# 약점 분석 결과
print("\n=== 약점 분석 결과 ===")
print(weakness_analyzer.generate_study_recommendation())

print("\n=== 키워드별 정답률 ===")
print(weakness_analyzer.analyze_by_keyword())

## Cell 9: 통합 실행 - 모의고사 생성 및 CBT

In [ ]:
class BigDataPassAI:
    """BigData-Pass AI 통합 시스템"""
    
    def __init__(self, base_dir: Path):
        self.base_dir = base_dir
        self.knowledge_core = None
        self.vector_manager = None
        self.pattern_analyzer = None
        self.question_generator = None
        self.cbt = CBTSystem()
        self.weakness_analyzer = WeaknessAnalyzer()
        self.dynamic_gen = DynamicGenerator()
        
    def initialize(self, use_api: bool = True):
        """시스템 초기화"""
        print("[1/4] Knowledge Core 초기화...")
        self.knowledge_core = KnowledgeCore(
            self.base_dir / "data" / "json",
            self.base_dir / "data" / "빅데이터암기키트.pdf"
        )
        questions = self.knowledge_core.load_exam_json()
        pdf_docs = self.knowledge_core.load_pdf()
        
        print("\n[2/4] Pattern Analyzer 초기화...")
        self.pattern_analyzer = PatternAnalyzer(questions)
        
        if use_api and "OPENAI_API_KEY" in os.environ:
            print("\n[3/4] Vector Store 초기화...")
            self.vector_manager = VectorStoreManager(
                persist_directory=str(self.base_dir / "chroma_db")
            )
            self.vector_manager.create_exam_vectorstore(questions)
            if pdf_docs:
                self.vector_manager.create_pdf_vectorstore(pdf_docs)
            
            print("\n[4/4] Question Generator 초기화...")
            self.question_generator = QuestionGenerator(
                self.vector_manager, 
                self.pattern_analyzer
            )
        else:
            print("\n[3/4] Vector Store 건너뜀 (API 키 없음)")
            print("[4/4] Question Generator 건너뜀 (API 키 없음)")
        
        print("\n시스템 초기화 완료!")
    
    def generate_exam(self, n_questions: int = 10, use_llm: bool = True) -> List[Dict]:
        """모의고사 생성"""
        questions = []
        
        if use_llm and self.question_generator:
            questions = self.question_generator.generate_mixed_exam(n_questions)
        else:
            # API 없이 계산 문제만 생성
            for i in range(n_questions):
                problem = self.dynamic_gen.generate_random_problem()
                choices = [problem["correct"]] + problem["wrong"][:3]
                random.shuffle(choices)
                answer = choices.index(problem["correct"]) + 1
                
                questions.append({
                    "id": f"GEN-{i+1:02d}",
                    "question": problem["question"],
                    "choices": choices,
                    "answer": answer,
                    "keywords": problem.get("keywords", []),
                    "subject": 4,
                    "type": "calculation",
                    "explanation": {
                        "problem_analysis": f"{problem['answer_type']} 문제",
                        "theory_basis": problem.get("calculation", ""),
                        "answer_derivation": f"정답: {problem['correct']}",
                        "wrong_analysis": "계산 방식 확인 필요"
                    }
                })
        
        return questions
    
    def start_cbt(self, questions: List[Dict]) -> TestSession:
        """CBT 세션 시작"""
        return self.cbt.create_session(questions)
    
    def export_exam_json(self, questions: List[Dict], filename: str):
        """생성된 문제를 JSON으로 내보내기"""
        export_data = {
            "exam_info": {
                "title": "BigData-Pass AI 생성 모의고사",
                "total_questions": len(questions),
                "source": "BigData-Pass AI"
            },
            "questions": questions
        }
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(export_data, f, ensure_ascii=False, indent=2)
        
        print(f"문제 내보내기 완료: {filename}")

In [ ]:
# BigData-Pass AI 시스템 초기화
bigdata_ai = BigDataPassAI(BASE_DIR)

# API 키 유무에 따라 초기화
use_api = "OPENAI_API_KEY" in os.environ
bigdata_ai.initialize(use_api=use_api)

In [ ]:
# 모의고사 생성
print("\n=== 모의고사 생성 ===")
exam_questions_gen = bigdata_ai.generate_exam(n_questions=5, use_llm=use_api)
print(f"생성된 문제 수: {len(exam_questions_gen)}개")

# 첫 번째 문제 미리보기
if exam_questions_gen:
    q = exam_questions_gen[0]
    print(f"\n[문제 1 미리보기]")
    print(f"문제: {q['question'][:100]}...")
    print(f"보기:")
    for i, c in enumerate(q['choices'], 1):
        print(f"  {i}. {c}")
    print(f"정답: {q['answer']}번")

In [ ]:
# 생성된 문제 JSON 내보내기
if exam_questions_gen:
    export_path = str(BASE_DIR / "data" / "json" / "generated_exam.json")
    bigdata_ai.export_exam_json(exam_questions_gen, export_path)

## Cell 10: 사용 가이드

In [ ]:
print("""
===============================================
BigData-Pass AI 사용 가이드
===============================================

[1] API 키 설정 (OpenAI)
    os.environ["OPENAI_API_KEY"] = "your-api-key"

[2] 시스템 초기화
    bigdata_ai = BigDataPassAI(BASE_DIR)
    bigdata_ai.initialize(use_api=True)

[3] 모의고사 생성
    questions = bigdata_ai.generate_exam(n_questions=20)

[4] CBT 시험 시작
    session = bigdata_ai.start_cbt(questions)

[5] 약점 분석
    bigdata_ai.weakness_analyzer.plot_analysis()

[6] 문제 내보내기
    bigdata_ai.export_exam_json(questions, "my_exam.json")

===============================================
API 없이 사용 가능한 기능
===============================================
- 계산 문제 생성 (혼동행렬, 통계, 확률, 표본추출)
- 기출문제 패턴 분석
- CBT 시스템
- 약점 분석

API 필요한 기능
===============================================
- RAG 기반 개념 문제 생성
- Vector Store 검색
- LLM 기반 해설 생성
""")